# 02 — Data preparation and baseline models

This notebook defines a reproducible holdout, a separate validation partition for baseline comparison, and runnable model code. It does not remove duplicate rows. The outer test partition remains untouched until model choices are frozen; cross-validation and hyperparameter tuning are the next phase.

Start with [Notebook 01](01_dataset_understanding_eda.ipynb) so the pinned CSV is available under `data/raw/`. The holdout keeps exact-feature duplicates together; it is a learning benchmark, not evidence of production generalization.


## Load data and define inputs

A conventional `train_test_split` with `stratify=y` assigns rows randomly while approximately preserving each class proportion in both partitions. This is a useful first pattern for imbalanced classification, but it does not keep duplicate feature signatures together.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    fbeta_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

working_directory = Path.cwd().resolve()
repository_candidates = (working_directory, *working_directory.parents)
REPO_ROOT = next(
    (candidate for candidate in repository_candidates
     if (candidate / 'README.md').is_file() and (candidate / 'notebooks').is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')
RAW_CSV = REPO_ROOT / 'data' / 'raw' / 'creditcard.csv'
if not RAW_CSV.is_file():
    raise FileNotFoundError('Run Notebook 01 to retrieve the pinned dataset before continuing.')

df = pd.read_csv(RAW_CSV)
X = df.drop(columns=['Class'])
y = df['Class']
print(f'Input rows: {len(X):,}; numeric inputs: {X.shape[1]}')


Input rows: 284,807; numeric inputs: 30


## Conventional stratified split: useful, but not enough here

`stratify=y` makes each partition's class proportions approximately match the full target distribution. It does not group identical feature rows. For the fixed conventional split below, the supplied diagnostics report 227,845 training rows and 56,962 test rows, with 394 and 98 frauds respectively. They also report 293 distinct exact-feature signatures present on both sides. The rows must not be deleted to hide this overlap.


In [2]:
X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X, y, test_size=.20, random_state=42, stratify=y
)

# Hash X only, not the label, to identify identical feature signatures.
feature_groups = pd.util.hash_pandas_object(X, index=False).astype('uint64')
random_train_groups = feature_groups.loc[X_train_random.index].unique()
random_test_groups = feature_groups.loc[X_test_random.index].unique()
random_overlap = np.intersect1d(random_train_groups, random_test_groups)
print(f'Conventional split rows: {len(X_train_random):,} train / {len(X_test_random):,} test')
print(f'Frauds: {int(y_train_random.sum())} train / {int(y_test_random.sum())} test')
print(f'Distinct exact-feature signatures crossing the split: {len(random_overlap)}')


Conventional split rows: 227,845 train / 56,962 test
Frauds: 394 train / 98 test
Distinct exact-feature signatures crossing the split: 293


## Reproducible group-aware holdout

The structural audit found 1,081 exact duplicate rows (0.3796%) and zero feature-conflicting-target groups. Do not delete duplicates: retain the observations, but assign rows with identical input features to the same partition.

The next cell hashes **X only**, so the target cannot influence group membership, then takes the first fold from `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)`. Stratification approximately preserves class proportions while grouping prevents an identical feature signature from appearing in both train and the outer test partition. The supplied diagnostic result is 227,845 development rows / 56,962 outer-test rows, with 393 / 99 frauds, rates 0.1725% / 0.1738%, and zero group overlap. The outer test set is reserved for one final evaluation after model choices are frozen.


In [3]:
group_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_indices, test_indices = next(
    group_splitter.split(X, y, groups=feature_groups.to_numpy())
)
X_train = X.iloc[train_indices].copy()
X_test = X.iloc[test_indices].copy()
y_train = y.iloc[train_indices].copy()
y_test = y.iloc[test_indices].copy()

train_feature_groups = feature_groups.iloc[train_indices].to_numpy()
test_feature_groups = feature_groups.iloc[test_indices].to_numpy()
group_overlap = np.intersect1d(np.unique(train_feature_groups), np.unique(test_feature_groups))
split_summary = pd.DataFrame({
    'rows': [len(X_train), len(X_test)],
    'frauds': [int(y_train.sum()), int(y_test.sum())],
    'fraud_rate_pct': [100 * y_train.mean(), 100 * y_test.mean()],
}, index=['train', 'test'])
print(f'Exact-feature group overlap: {len(group_overlap)}')
assert len(group_overlap) == 0
split_summary


Exact-feature group overlap: 0


,rows,frauds,fraud_rate_pct
train,227845,393,0.172486
test,56962,99,0.173800


### Split interpretation

The outer grouped holdout keeps all rows, including exact duplicates, but places each exact-feature signature wholly in one partition. It therefore answers a stricter question than a row-random split: how does a model behave on held-out feature signatures? A group-aware split does not solve the dataset's short historical time span or establish production performance. The observed fold counts and rates above are diagnostic reference values, not model results.


## Create a validation partition without touching the outer test set

Comparing several models on the outer test set would use that test set to choose a model. Instead, make a second group-aware split inside the development partition. Use this validation subset for the first baseline comparison; leave `X_test` and `y_test` unused until the model and threshold are frozen. Identical feature signatures remain together in this inner split too.


In [4]:
validation_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=43)
fit_indices, validation_indices = next(
    validation_splitter.split(X_train, y_train, groups=train_feature_groups)
)
X_fit = X_train.iloc[fit_indices].copy()
X_validation = X_train.iloc[validation_indices].copy()
y_fit = y_train.iloc[fit_indices].copy()
y_validation = y_train.iloc[validation_indices].copy()
fit_groups = train_feature_groups[fit_indices]
validation_groups = train_feature_groups[validation_indices]
inner_group_overlap = np.intersect1d(np.unique(fit_groups), np.unique(validation_groups))
assert len(inner_group_overlap) == 0
validation_summary = pd.DataFrame({
    'rows': [len(X_fit), len(X_validation)],
    'frauds': [int(y_fit.sum()), int(y_validation.sum())],
    'fraud_rate_pct': [100 * y_fit.mean(), 100 * y_validation.mean()],
}, index=['model training', 'validation'])
print(f'Exact-feature groups crossing the inner split: {len(inner_group_overlap)}')
validation_summary


Exact-feature groups crossing the inner split: 0


,rows,frauds,fraud_rate_pct
model training,182276,314,0.172266
validation,45569,79,0.173363


## Preprocessing and SMOTE

`Time`, `Amount`, and the anonymized PCA features are all numeric. Their raw scales differ substantially: the supplied profile reports standard deviation 1.9587 for V1 and 0.3301 for V28, while `Time` and `Amount` have much larger raw scales. Standardize **all 30 numeric inputs**, not just `Time` and `Amount`, for the MLP and especially for SMOTE's nearest-neighbor distance calculation. Never scale the target `Class`.

Trees do not intrinsically require standardized features, but SMOTE does use distances. Put `StandardScaler` and `SMOTE(random_state=42)` inside an `imblearn.pipeline.Pipeline` so the scaler is fit on each model-training partition and synthetic samples are generated only from that partition. Validation data keep the original class prevalence and are transformed by the fitted scaler without resampling. The outer test partition remains untouched. In future cross-validation, pass this complete pipeline to the CV routine so each fold fits its own scaler and resampler.

SMOTE runs only when the baseline training cell is executed. Neither the validation nor the outer-test partition is resampled.


## Baseline model definitions

These are intentionally simple reference classifiers. The following cell only creates estimator objects; it does not fit them.


In [5]:
models = {
    'Decision tree': DecisionTreeClassifier(random_state=42),
    'Random forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'AdaBoost': AdaBoostClassifier(random_state=42),
}


## One holdout training and evaluation loop

The next cell is runnable to compare the three baselines on the untouched validation partition. For each baseline the pipeline fits the scaler, applies SMOTE to `X_fit` and `y_fit` only, and fits the classifier. It then scores the naturally imbalanced validation data. Average Precision summarizes the precision-recall curve; precision, recall, and F2 use each estimator's default prediction threshold. The confusion matrix uses rows = true class and columns = predicted class, ordered `[0, 1]`. Accuracy is intentionally omitted.

This is a first validation-based baseline, not cross-validation or hyperparameter tuning. Those are deferred to the next phase. Do not use the outer test set to choose among these models or tune thresholds. In future CV, pass the complete imbalanced-learn pipeline and group labels so SMOTE cannot sample across a validation fold and exact-feature duplicates stay together.


In [6]:
metric_rows = []
for model_name, classifier in models.items():
    pipeline = ImbPipeline(steps=[
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('classifier', classifier),
    ])
    pipeline.fit(X_fit, y_fit)
    y_score = pipeline.predict_proba(X_validation)[:, 1]
    y_pred = pipeline.predict(X_validation)

    metric_rows.append({
        'model': model_name,
        'Average Precision (PR-AUC summary)': average_precision_score(y_validation, y_score),
        'precision': precision_score(y_validation, y_pred, zero_division=0),
        'recall': recall_score(y_validation, y_pred, zero_division=0),
        'F2': fbeta_score(y_validation, y_pred, beta=2, zero_division=0),
    })
    print(f'{model_name} validation confusion matrix (rows=true, columns=predicted; labels 0, 1):')
    print(confusion_matrix(y_validation, y_pred, labels=[0, 1]))

baseline_summary = pd.DataFrame(metric_rows).set_index('model')
baseline_summary


Decision tree validation confusion matrix (rows=true, columns=predicted; labels 0, 1):
[[45383   107]
 [   19    60]]


Random forest validation confusion matrix (rows=true, columns=predicted; labels 0, 1):
[[45481     9]
 [   15    64]]


AdaBoost validation confusion matrix (rows=true, columns=predicted; labels 0, 1):
[[44067  1423]
 [    7    72]]


,Average Precision (PR-AUC summary),precision,recall,F2
model,,,,
Decision tree,0.273289,0.359281,0.759494,0.621118
Random forest,0.849946,0.876712,0.810127,0.822622
AdaBoost,0.736748,0.048161,0.911392,0.198785


## First validation results and interpretation

The validation partition contains 45,569 transactions, including 79 frauds (0.1734%). At the classifiers' default 0.5 threshold, the Decision Tree detected 60 of 79 frauds (recall 0.7595) with 107 false alerts; Random Forest detected 64 (recall 0.8101) with 9 false alerts; AdaBoost detected 72 (recall 0.9114) but produced 1,423 false alerts.

Random Forest is the strongest provisional candidate on this split: Average Precision is 0.8499 and F2 is 0.8226, the highest of the three. AdaBoost demonstrates why recall alone is insufficient: it catches the most fraud here, but its precision is only 0.0482 and its F2 is 0.1988. The Decision Tree is a simple reference point, with Average Precision 0.2733 and F2 0.6211.

These are preliminary results from one validation partition with only 79 fraud examples. The threshold was not tuned, cross-validation has not been run, and the outer test set has not been evaluated. Do not describe this as the final model or as production performance.


## Proposed MLP design — hypothesis, not a trained result

The compact network size is a starting hypothesis for 30 tabular numeric inputs and only 492 fraud observations, not a claim that this architecture is optimal. Use binary log-loss (cross-entropy), ReLU hidden units, L2 regularization through `alpha`, and the Adam optimizer. Starting settings are hidden layers `(64, 32)`, `learning_rate_init=1e-3`, `alpha=1e-4`, `batch_size=256`, `max_iter=100`, and random seed 42. These settings must be validated, not assumed optimal.

Do not enable `early_stopping` inside the post-SMOTE estimator: it would create an internal validation split after resampling rather than provide the intended natural-prevalence validation. Validation and CV design belong to the next phase. There are no categorical features, card/customer IDs, or behavioral sequences, so embeddings and sequential networks have no supported input structure here.


In [7]:
mlp_hypothesis = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    learning_rate_init=1e-3,
    alpha=1e-4,
    batch_size=256,
    max_iter=100,
    random_state=42,
)


## What remains for the next phase

- After selecting a baseline on validation, refit it on the development data and evaluate the outer test set once; keep that result as the final holdout estimate.
- Design cross-validation and hyperparameter tuning explicitly; keep all preprocessing and SMOTE inside each training fold and preserve duplicate groups.
- Tune decision thresholds against validation predictions at natural prevalence and report fraud recall alongside false-alert burden.
- Add the proposed MLP as a separate experiment and compare it with the baselines without treating its initial settings as optimal.
- State the two-day 2013 data limitation in every performance summary; this holdout cannot establish current, cross-issuer, or production generalization.
